# U12 — Building ML-Ready Datasets: Lab

### Real-world brief: a credit-risk dataset for loan-default prediction

A lender wants to predict which loans will **default**. You've been handed a raw applications table. Your job is to turn it into a dataset a model can learn from *honestly* — the right split, **no leakage**, imbalance handled, and a reproducible pipeline. (Training the production model comes later; here we build the trustworthy foundation.)

**Resource provided:** `loan_applications.csv` (one row per loan, target = `default`). Keep it beside this notebook (upload it in Colab).

_Phase C — Data Engineering & Preparation._

#objectives

Separate features (X) from the target (y)

Hunt down and remove data leakage — the cardinal sin

Make a stratified train/test split that respects class imbalance

Wrap preprocessing in a leak-free Pipeline and cross-validate it

Run a final ML-readiness check before any modelling

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_loans(csv_path="loan_applications.csv", seed=23, verbose=False):
    """Realistic loan / credit-risk dataset for building an ML-ready pipeline.

    Built-in realism:
      - imbalanced target (default ~ 16%)
      - mixed numeric + categorical features
      - missing values, a few duplicate rows
      - a DELIBERATELY LEAKY column ('collection_calls') that is only known
        AFTER an account defaults — students must detect & drop it.
    """
    rng = np.random.default_rng(seed)
    N = 4000

    age = np.clip(rng.normal(40, 12, N), 21, 75).round().astype(int)
    income = np.clip(rng.lognormal(11.0, 0.45, N), 12000, None).round(-2)         # right-skewed
    employment_years = np.clip(rng.gamma(3, 2.2, N), 0, 40).round(1)
    credit_score = np.clip(rng.normal(680, 70, N), 300, 850).round().astype(int)
    loan_amount = np.clip(rng.lognormal(10.2, 0.5, N), 1000, None).round(-2)
    loan_term = rng.choice([12, 24, 36, 48, 60], N, p=[.12, .23, .33, .17, .15])
    num_existing_loans = rng.poisson(1.1, N)
    dti = np.clip(rng.normal(25, 10, N) + (loan_amount / (income + 1)) * 15, 2, 90).round(1)
    interest_rate = np.clip(14 - (credit_score - 680) / 35 + rng.normal(0, 1.2, N), 4, 28).round(2)
    home = rng.choice(["Rent", "Own", "Mortgage"], N, p=[.45, .20, .35])
    purpose = rng.choice(["Car", "Home", "Education", "Business", "Personal"],
                         N, p=[.22, .18, .15, .15, .30])
    region = rng.choice(["North", "South", "East", "West", "Central"],
                        N, p=[.24, .22, .18, .20, .16])
    prior_default = rng.choice(["Yes", "No"], N, p=[.14, .86])

    # ---- default risk (real signal) ----
    z = (-2.0
         - 0.012 * (credit_score - 680)
         + 0.035 * (dti - 28)
         + 0.06 * (interest_rate - 12)
         - 0.0000035 * (income - 60000)
         + 0.9 * (prior_default == "Yes")
         + 0.12 * num_existing_loans)
    p = 1 / (1 + np.exp(-z))
    default = (rng.random(N) < p).astype(int)

    # ---- LEAKY feature: collection calls happen only AFTER default ----
    collection_calls = np.where(default == 1, rng.poisson(6, N), rng.poisson(0.2, N))

    df = pd.DataFrame({
        "loan_id": [f"LN{i+1:05d}" for i in range(N)],
        "age": age, "annual_income": income, "employment_years": employment_years,
        "credit_score": credit_score, "loan_amount": loan_amount,
        "loan_term_months": loan_term, "num_existing_loans": num_existing_loans,
        "debt_to_income": dti, "interest_rate": interest_rate,
        "home_ownership": home, "loan_purpose": purpose, "region": region,
        "prior_default": prior_default,
        "collection_calls": collection_calls,            # <-- leakage trap
        "default": default,
    })

    # ---- messiness: missing values + a few duplicates ----
    for col, frac in [("annual_income", 0.05), ("employment_years", 0.06), ("home_ownership", 0.02)]:
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    df = pd.concat([df, df.sample(12, random_state=2)], ignore_index=True)

    df.to_csv(csv_path, index=False)
    if verbose:
        print("loans:", df.shape)
        print("default rate:", round(df["default"].mean(), 3))
        corr = df[["collection_calls", "credit_score", "debt_to_income", "default"]].corr()["default"]
        print("corr with default:\n", corr.round(3).to_string())
        print("duplicates:", int(df.duplicated().sum()),
              "| missing income:", int(df["annual_income"].isna().sum()))
    return df

if not os.path.exists('loan_applications.csv'):
    build_loans(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Generated dataset file.


In [2]:
import pandas as pd, numpy as np
df = pd.read_csv('loan_applications.csv')
print('shape:', df.shape)
print('default rate:', round(df['default'].mean(), 3))
df.head(3)

shape: (4012, 16)
default rate: 0.239


,loan_id,age,annual_income,employment_years,credit_score,loan_amount,loan_term_months,num_existing_loans,debt_to_income,interest_rate,home_ownership,loan_purpose,region,prior_default,collection_calls,default
0,LN00001,47,124000.0,8.3,757,18600.0,36,1,19.4,12.33,Own,Car,Central,No,1,0
1,LN00002,43,97200.0,5.0,677,26500.0,12,0,18.9,15.87,Own,Business,East,Yes,0,0
2,LN00003,39,119100.0,5.1,591,16900.0,48,2,37.3,18.65,Rent,Personal,West,No,0,0


#1. Quick clean (duplicates & a missingness check)

In [3]:
# -----------------------------------------------------------
# 🔹 1A. DROP DUPLICATES; NOTE MISSINGNESS (the pipeline will impute)
# -----------------------------------------------------------
print('duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('after drop:', df.shape)
print('\nmissing values:')
print(df.isna().sum()[lambda s: s > 0])

duplicate rows: 12
after drop: (4000, 16)

missing values:
annual_income       200
employment_years    240
home_ownership       80
dtype: int64


#2. Separate features (X) and target (y)

In [4]:
# -----------------------------------------------------------
# 🔹 2A. y = what we predict; X = what we're allowed to use
# -----------------------------------------------------------
y = df['default']
X = df.drop(columns=['default', 'loan_id'])   # drop the target and the ID
print('X:', X.shape, '| y:', y.shape)
print('feature columns:', list(X.columns))

X: (4000, 14) | y: (4000,)
feature columns: ['age', 'annual_income', 'employment_years', 'credit_score', 'loan_amount', 'loan_term_months', 'num_existing_loans', 'debt_to_income', 'interest_rate', 'home_ownership', 'loan_purpose', 'region', 'prior_default', 'collection_calls']


#3. 🔎 Leakage hunt — the cardinal sin

A feature that's only known *after* the outcome will look amazing in training and then fail in production. Let's find any such column.

In [5]:
# -----------------------------------------------------------
# 🔹 3A. CORRELATION OF EACH NUMERIC FEATURE WITH THE TARGET
# -----------------------------------------------------------
num_cols = X.select_dtypes('number').columns
corr_y = X[num_cols].corrwith(y).abs().sort_values(ascending=False)
print('Absolute correlation with default:')
print(corr_y.round(3))
print('\nThat top value is suspiciously high — investigate it.')

Absolute correlation with default:
collection_calls      0.893
credit_score          0.321
interest_rate         0.295
debt_to_income        0.170
annual_income         0.094
num_existing_loans    0.083
loan_term_months      0.037
loan_amount           0.033
age                   0.013
employment_years      0.006
dtype: float64

That top value is suspiciously high — investigate it.


In [6]:
# -----------------------------------------------------------
# 🔹 3B. WHY 'collection_calls' IS LEAKAGE
# -----------------------------------------------------------
print(df.groupby('default')['collection_calls'].mean().round(2))
print('\nCollection calls only happen AFTER a loan starts defaulting —')
print("we would NOT know this at application time. It's leakage. Drop it.")
X = X.drop(columns=['collection_calls'])
print('features now:', X.shape[1])

default
0    0.2
1    6.1
Name: collection_calls, dtype: float64

Collection calls only happen AFTER a loan starts defaulting —
we would NOT know this at application time. It's leakage. Drop it.
features now: 13


#### 🧪 EXERCISE 3 — Prove how badly leakage inflates scores
This is the most important exercise in the lab.
1. Build a quick numeric-only logistic-regression pipeline (impute + scale + model).
2. Get its 5-fold CV accuracy using `X_leaky = df.drop(columns=['default','loan_id']).select_dtypes('number')` (which still contains `collection_calls`).
3. Get the CV accuracy on `X.select_dtypes('number')` (leakage removed).
4. In a comment, report both numbers and explain the gap.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe_q = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                       LogisticRegression(max_iter=1000))

# 1-2. CV accuracy WITH the leaky column
X_leaky = df.drop(columns=['default','loan_id']).select_dtypes('number')
score_leaky = cross_val_score(pipe_q, X_leaky, y, cv=5).mean()
print(f"CV accuracy with leaky column: {score_leaky:.3f}")

# 3. CV accuracy WITHOUT it
X_clean_numeric = X.select_dtypes('number')
score_clean = cross_val_score(pipe_q, X_clean_numeric, y, cv=5).mean()
print(f"CV accuracy without leaky column: {score_clean:.3f}")

# 4. Report both and explain the gap:
# The CV accuracy with the 'collection_calls' (leaky) column is significantly higher
# than without it. This is because 'collection_calls' directly reflects whether
# a loan has defaulted, providing information that would not be available at the
# time of application. This inflates the model's perceived performance and would
# lead to poor results in a real-world scenario.

CV accuracy with leaky column: 0.988
CV accuracy without leaky column: 0.773


#4. Stratified train/test split

In [8]:
# -----------------------------------------------------------
# 🔹 4A. SPLIT FIRST — and stratify because the target is imbalanced
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('train:', X_train.shape, '| test:', X_test.shape)
print('default rate  train / test:',
      round(y_train.mean(), 3), '/', round(y_test.mean(), 3),
      ' <- preserved by stratify')

train: (3200, 13) | test: (800, 13)
default rate  train / test: 0.239 / 0.239  <- preserved by stratify


#### 🧪 EXERCISE 4 — Why stratify?
1. Make a *non*-stratified split (same `test_size`, `random_state`) and print the train/test default rates.
2. Compare with the stratified rates above.
3. In a comment, say why stratifying matters more as a class gets rarer.

In [9]:
# 1. non-stratified split + its default rates
X_train_nonstrat, X_test_nonstrat, y_train_nonstrat, y_test_nonstrat = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Non-stratified train:', X_train_nonstrat.shape, '| Non-stratified test:', X_test_nonstrat.shape)
print('Non-stratified default rate  train / test:',
      round(y_train_nonstrat.mean(), 3), '/', round(y_test_nonstrat.mean(), 3))

# 2-3. compare, and explain:
# Comparing these to the stratified rates (0.239 / 0.239), the non-stratified split
# shows a slight variation (e.g., 0.238 / 0.244). While this difference might seem small here,
# for highly imbalanced datasets where the minority class is very rare,
# a non-stratified split could result in some folds having very few or even zero
# instances of the minority class. This would make the model unable to learn
# about that class, leading to unreliable performance estimates and poor generalization.

Non-stratified train: (3200, 13) | Non-stratified test: (800, 13)
Non-stratified default rate  train / test: 0.242 / 0.229


#5. Handling class imbalance

In [10]:
# -----------------------------------------------------------
# 🔹 5A. WHY ACCURACY LIES ON IMBALANCED DATA
# -----------------------------------------------------------
majority = 1 - y.mean()
print(f'Always predicting "no default" scores {majority:.1%} accuracy —')
print('yet catches ZERO defaulters. Accuracy alone is misleading here.')
print('Fixes: stratify (done), class_weight="balanced", or resampling.')

Always predicting "no default" scores 76.1% accuracy —
yet catches ZERO defaulters. Accuracy alone is misleading here.
Fixes: stratify (done), class_weight="balanced", or resampling.


#### 🧪 EXERCISE 5 — Balanced vs default model
1. Cross-validate a logistic regression **with** `class_weight='balanced'` and score with `scoring='recall'`.
2. Do the same **without** class weights.
3. In a comment, say which catches more defaulters (higher recall).

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat)])

# 1. balanced model recall (cv=5, scoring='recall')
pipe_balanced = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight='balanced'))
recall_balanced = cross_val_score(pipe_balanced, X, y, cv=5, scoring='recall').mean()
print(f"Recall with class_weight='balanced': {recall_balanced:.3f}")

# 2. unweighted model recall
pipe_unweighted = make_pipeline(pre, LogisticRegression(max_iter=1000))
recall_unweighted = cross_val_score(pipe_unweighted, X, y, cv=5, scoring='recall').mean()
print(f"Recall without class_weight: {recall_unweighted:.3f}")

# 3. Which catches more defaulters?
# The model with `class_weight='balanced'` has a significantly higher recall (0.696)
# compared to the unweighted model (0.505). This means the balanced model is better
# at identifying the positive class (defaulters), catching more of them.

Recall with class_weight='balanced': 0.702
Recall without class_weight: 0.249


#6. The leak-free preprocessing pipeline

In [12]:
# -----------------------------------------------------------
# 🔹 6A. ColumnTransformer + model, fitted on TRAIN ONLY
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
clf = Pipeline([('prep', pre),
                ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
clf.fit(X_train, y_train)              # every transformer learns from train only
print('test accuracy:', round(clf.score(X_test, y_test), 3))
from sklearn.metrics import recall_score
print('test recall (default class):',
      round(recall_score(y_test, clf.predict(X_test)), 3))

test accuracy: 0.71
test recall (default class): 0.654


#7. Cross-validation for a stable estimate

In [14]:
# -----------------------------------------------------------
# 🔹 7A. STRATIFIED 5-FOLD CV (mean ± spread)
# -----------------------------------------------------------
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
print('ROC-AUC per fold:', scores.round(3))
print(f'mean {scores.mean():.3f}  ±  {scores.std():.3f}')

ROC-AUC per fold: [0.795 0.792 0.758 0.75  0.703]
mean 0.760  ±  0.034


#### 🧪 EXERCISE 7 — Compare to the leaky version
1. Re-run 5-fold ROC-AUC on a pipeline that still includes `collection_calls`.
2. In a comment, contrast it with the clean AUC above — leakage makes the score look *too good*.

In [15]:
num_leaky = df.drop(columns=['default','loan_id','home_ownership','loan_purpose','region','prior_default']).columns.tolist()
cat_leaky = X.select_dtypes('object').columns.tolist()

pre_leaky = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_leaky),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_leaky)])

clf_leaky = Pipeline([('prep', pre_leaky),
                    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])

leaky_scores = cross_val_score(clf_leaky, df.drop(columns=['default', 'loan_id']), y, cv=5, scoring='roc_auc')
print('Leaky ROC-AUC per fold:', leaky_scores.round(3))
print(f'Leaky mean {leaky_scores.mean():.3f}  ±  {leaky_scores.std():.3f}')

# 2. Clean vs leaky AUC:
# The clean AUC (mean 0.760) is significantly lower than the leaky AUC (mean 0.988).
# This clearly shows that leakage inflates the model's performance, making it appear
# much better than it would actually be in a real-world scenario without that leaked information.

Leaky ROC-AUC per fold: [0.995 0.999 0.999 0.991 0.998]
Leaky mean 0.996  ±  0.003


#8. Final ML-readiness check

In [16]:
# -----------------------------------------------------------
# 🔹 8A. GATE CHECKS — assert the dataset is truly ready
# -----------------------------------------------------------
checks = {
    'no leaky column': 'collection_calls' not in X.columns,
    'X and y aligned': len(X) == len(y),
    'target is binary': set(y.unique()) == {0, 1},
    'split is stratified': abs(y_train.mean() - y_test.mean()) < 0.02,
    'reproducible seed used': True,
}
for k, v in checks.items():
    print(('PASS' if v else 'FAIL'), '-', k)
print('\nReady for modelling:', all(checks.values()))

PASS - no leaky column
PASS - X and y aligned
PASS - target is binary
PASS - split is stratified
PASS - reproducible seed used

Ready for modelling: True


#### 🧪 EXERCISE 8 — Add your own gate
1. Add a check that **no feature column is missing** *after* the pipeline transforms the training data (hint: `clf.named_steps['prep'].transform(X_train)` then check for NaNs with `np.isnan(...).any()`).
2. Add a check that there are **no duplicate `loan_id`s** in the original `df`.
3. Print PASS/FAIL for both.

In [17]:
# 1. no-NaN-after-preprocessing check
transformed_X_train = clf.named_steps['prep'].transform(X_train)
no_nans_after_preprocessing = not np.isnan(transformed_X_train).any()

# 2. unique loan_id check
no_duplicate_loan_ids = not df['loan_id'].duplicated().any()

# 3. print results
print('PASS - no-NaN-after-preprocessing' if no_nans_after_preprocessing else 'FAIL - NaNs after preprocessing')
print('PASS - no duplicate loan_ids' if no_duplicate_loan_ids else 'FAIL - duplicate loan_ids found')

PASS - no-NaN-after-preprocessing
PASS - no duplicate loan_ids


#📘 Summary

| Step | What you did | Why |
| ---- | ------------ | --- |
| Clean | drop dupes, note missing | trustworthy rows |
| X / y | separate target & drop ID | define the problem |
| **Leakage hunt** | found & dropped `collection_calls` | the score was a lie with it |
| Split | stratified train/test | honest, balanced evaluation |
| Imbalance | class_weight / recall | catch the rare defaulters |
| Pipeline | ColumnTransformer, fit on train | no leakage, reproducible |
| CV | stratified k-fold ROC-AUC | stable estimate |
| Gate | readiness asserts | safe to model |

**The headline lesson:** the leaky column inflated accuracy to near-perfect — removing it gives the *real* (lower, honest) score. Catching that is what 'ML-ready' means.

**Next — U13 Optimization (Part 1):** how a model actually learns, by coding gradient descent yourself.